# Imports & Definitions

In [28]:
import config
import os
os.environ["OPENAI_API_KEY"] = config.OPENAI_API_KEY
os.environ["OPENAI_API_BASE"] = config.OPENAI_API_BASE

In [2]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.output_parsers import PydanticOutputParser

from pydantic import BaseModel, PositiveInt, PositiveFloat, TypeAdapter
from typing import List

import json
import ast
import pandas as pd

# Prompt Definition

In [9]:
class HousingListing(BaseModel):
    id: int
    neighborhood_name: str
    price: PositiveInt
    bedrooms: PositiveInt
    bathrooms: PositiveFloat
    house_size: PositiveInt
    year_built: PositiveInt
    description: str
    neighborhood_description: str

parser = PydanticOutputParser(pydantic_object=HousingListing)

In [20]:
instructions = """
You are a housing market expert.
Generate a list of dummy housing listings. Each listing should be a JSON object with the following fields:
- id (integer)
- neighborhood name (string, a fictional neighborhood name)
- price (integer, between 100,000 and 1,000,000 US dollars)
- bedrooms (integer, between 1 and 6)
- bathrooms (float, between 1 and 4, increments of 0.5)
- house_size (integer, between 1000 and 10000 square feet)
- year_built (integer, between 1950 and 2023)
- description (string, a short description of the property)
- neighborhood description (string, a short description of the neighborhood)
"""

In [21]:
example_template = """
                ID: {id}
                Neighborhood Name: {neighborhood_name}
                Price: ${price}
                Bedrooms: {bedrooms}
                Bathrooms: {bathrooms}
                House Size: {house_size} sqft
                Year Built: {year_built}
                Description: {description}
                Neighborhood Description: {neighborhood_description}
                """

In [22]:
examples = [{
    "id": 1,
    "neighborhood_name": "Green Oaks",
    "price": 800000,
    "bedrooms": 3,
    "bathrooms": 2,
    "house_size": 2000,
    "year_built": 1989,
    "description": "Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.",
    "neighborhood_description": "Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze."
}]

In [23]:
example_prompt = PromptTemplate(
    template=example_template,
    input_variables=[
        "id", "neighborhood_name", "price", "bedrooms", "bathrooms", "house_size",
         "year_built", "description", "neighborhood_description"
    ]
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=instructions +
    """
    
    Here are some examples:
    """,
    suffix="""
    Now, generate {batch_size} more listings in the same format. Return them as a valid JSON array
    {format_instructions}
    """,
    input_variables=["batch_size"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create Data

In [3]:
model_name = "gpt-3.5-turbo"
temperature = 1.0
llm = ChatOpenAI(model_name=model_name, temperature=temperature)

In [25]:
def create_data(llm, batch_size=5):

    # Run the LangChain Model
    response = llm.invoke(few_shot_prompt.format(batch_size=batch_size))
    # Convert string into python output
    try:
        listings_ast = ast.literal_eval(response.content)  # Converts to Python list of dicts
    except (SyntaxError, ValueError) as e:
        print("Error parsing response with ast.literal_eval:", e)
        listings_ast = []

    # Validate that responses are according to expected output
    try:
        validated_listings = TypeAdapter(List[HousingListing]).validate_python(listings_ast)
    
        # Convert to Pandas DataFrame
        df = pd.DataFrame([listing.model_dump() for listing in validated_listings])
    
        return df
    
    except Exception as e:
        print("Error parsing with Pydantic:", e)
        return pd.DataFrame()

In [26]:
number_of_listings = 10
listings_dfs = []
batch_size = 10
while batch_size * len(listings_dfs) < number_of_listings:
    response_df = create_data(llm, batch_size)
    if not response_df.empty:
        listings_dfs.append(response_df)

In [28]:
df = pd.concat(listings_dfs)

In [33]:
df.head()

,id,neighborhood_name,price,bedrooms,bathrooms,house_size,year_built,description,neighborhood_description
0,2,Willow Creek,650000,4,3.5,3000,2005,Discover the perfect family home in the desira...,Willow Creek is known for its top-rated school...
1,3,Riverfront Estates,950000,5,4.0,5000,2010,Luxury living awaits in the prestigious Riverf...,"Riverfront Estates offers waterfront views, pr..."
2,4,Sunny Hills,400000,3,2.5,1800,1998,Welcome home to the charming neighborhood of S...,Sunny Hills is a family-friendly community wit...
3,5,Oak Ridge,750000,4,3.0,2800,2008,Live the ultimate suburban lifestyle in the so...,"Oak Ridge boasts tree-lined streets, community..."
4,6,Maple Grove,550000,3,2.5,2400,2000,Step into this meticulously maintained home in...,"Maple Grove offers tree-lined streets, communi..."


In [30]:
df[df['neighborhood_name']=='Willow Creek']['neighborhood_description'].values

array(['Willow Creek is known for its top-rated schools, tree-lined streets, and family-friendly atmosphere. Enjoy weekends at the nearby community pool or take a leisurely stroll to the quaint Willow Creek Town Center for shopping and dining.'],
      dtype=object)

In [31]:
df[['neighborhood_name','bedrooms','bathrooms','house_size']].values

array([['Willow Creek', 4, 3.5, 3000],
       ['Riverfront Estates', 5, 4.0, 5000],
       ['Sunny Hills', 3, 2.5, 1800],
       ['Oak Ridge', 4, 3.0, 2800],
       ['Maple Grove', 3, 2.5, 2400],
       ['Pinecrest Heights', 2, 1.5, 1500],
       ['Meadowbrook Ridge', 4, 3.0, 3200],
       ['Cedar Grove', 3, 2.0, 2200],
       ['Sunset Hills', 4, 3.5, 2600],
       ['Riverside Park', 3, 2.5, 1900]], dtype=object)

In [34]:
df.to_csv('dummy_housing_data_v2.csv')